# Part 2 - Exploratory Data Analysis (EDA)
Run Part 1 first or load the datasets again.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

TARGET='addicted_label'

# Uncomment if needed
# train = pd.read_csv('/kaggle/input/playground-series-s6e8/train.csv')


## Dataset Shape

In [ ]:
print("Rows :", train.shape[0])
print("Columns :", train.shape[1])

display(train.head())


## Target Distribution

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x=TARGET,data=train)
plt.title("Target Distribution")
plt.show()

print(train[TARGET].value_counts(normalize=True))


## Missing Values

In [ ]:
missing=train.isnull().sum().sort_values(ascending=False)
missing=missing[missing>0]

display(missing.to_frame("Missing"))

if len(missing)>0:
    plt.figure(figsize=(8,max(3,len(missing)*0.4)))
    missing.plot(kind='barh')
    plt.title("Missing Values")
    plt.show()


## Numerical Feature Distributions

In [ ]:
num_cols=train.select_dtypes(include=['number']).columns.tolist()
num_cols=[c for c in num_cols if c!=TARGET and c!='id']

for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.histplot(train[col],kde=True,bins=30)
    plt.title(col)
    plt.show()


## Numerical Features vs Target

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(data=train,x=TARGET,y=col)
    plt.title(col)
    plt.show()


## Categorical Features

In [ ]:
cat_cols=train.select_dtypes(include=['object','category']).columns.tolist()

for col in cat_cols:
    plt.figure(figsize=(7,4))
    sns.countplot(data=train,x=col,hue=TARGET)
    plt.xticks(rotation=30)
    plt.title(col)
    plt.tight_layout()
    plt.show()


## Correlation Heatmap

In [ ]:
corr=train[num_cols+[TARGET]].corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr,cmap='coolwarm',center=0)
plt.title("Correlation Matrix")
plt.show()


## Outlier Detection

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,2.5))
    sns.boxplot(x=train[col])
    plt.title(col)
    plt.show()


## Quick Feature Importance (CatBoost)

In [ ]:
from catboost import CatBoostClassifier

X=train.drop(columns=['id',TARGET])
y=train[TARGET]

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

for col in cat_cols:
    X[col]=X[col].fillna('Missing')

for col in X.columns:
    if col not in cat_cols:
        X[col]=X[col].fillna(X[col].median())

cat_idx=[X.columns.get_loc(c) for c in cat_cols]

model=CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    verbose=False,
    random_seed=42
)

model.fit(X,y,cat_features=cat_idx)

fi=pd.DataFrame({
    'Feature':X.columns,
    'Importance':model.get_feature_importance()
}).sort_values('Importance',ascending=False)

display(fi)

plt.figure(figsize=(8,6))
sns.barplot(data=fi.head(15),x='Importance',y='Feature')
plt.title("Top 15 Features")
plt.show()


### Observations

Fill this section with insights before moving to feature engineering.